# Variational Classifier — Ising phase classification

VQC binario (2 salidas) con la misma prep PCA→4.


## Librerias


In [ ]:
import json, time
from pathlib import Path
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import NesterovMomentumOptimizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
np.random.seed(123)


## Data


In [ ]:
# Load PennyLane qspin Ising (4 qubits) and build phase labels
# Features = ground-state amplitudes (Re||Im). Labels from physics: y=(h > |J|).
# IMPORTANT: h is NOT included as a feature (would leak the label).

ISING_LAYOUTS = [
    {"periodicity": "open", "lattice": "chain", "layout": "1x4"},
    {"periodicity": "closed", "lattice": "chain", "layout": "1x4"},
]


def fix_global_phase(states: np.ndarray) -> np.ndarray:
    """Remove global phase so the first amplitude is real and non-negative when possible."""
    ref = states[:, 0].copy()
    tiny = np.abs(ref) < 1e-12
    if np.any(tiny):
        idx = np.argmax(np.abs(states[tiny]), axis=1)
        ref[tiny] = states[tiny, idx]
        ref[np.abs(ref) < 1e-12] = 1.0
    return states * np.exp(-1j * np.angle(ref))[:, None]


def load_ising_phase_dataset(layouts=ISING_LAYOUTS):
    Xs, ys, meta_h = [], [], []
    for cfg in layouts:
        ds = qml.data.load(
            "qspin",
            sysname="Ising",
            periodicity=cfg["periodicity"],
            lattice=cfg["lattice"],
            layout=cfg["layout"],
        )[0]
        gs = np.array(ds.ground_states, dtype=complex)
        gs = fix_global_phase(gs)
        X = np.concatenate([gs.real, gs.imag], axis=1)  # (n, 32) for 4 qubits
        h = np.array(ds.parameters["h"], dtype=float)
        J = abs(float(ds.parameters["J"]))
        y = (h > J).astype(int)  # 0 ordered (low h), 1 disordered (high h)
        Xs.append(X)
        ys.append(y)
        meta_h.append(h)
        print(
            f"Loaded Ising {cfg['periodicity']} {cfg['lattice']} {cfg['layout']}: "
            f"X={X.shape}, classes={np.unique(y, return_counts=True)}, hc=|J|={J}"
        )
    X = np.vstack(Xs)
    y = np.concatenate(ys)
    h_all = np.concatenate(meta_h)
    return X, y, h_all


print("Loading Ising phase-classification dataset from PennyLane qspin...")
X_all, y_all, h_all = load_ising_phase_dataset()
print(f"Combined: n={len(X_all)}, features={X_all.shape[1]}, classes={np.unique(y_all, return_counts=True)}")


## Data Preparation


In [ ]:
n_qubits = 4
n_classes = 2
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    np.array(X_all, dtype=float),
    np.array(y_all, dtype=int),
    test_size=0.4, random_state=123, stratify=y_all,
)
print(f"Custom Dataset Loaded: Train={len(X_train_raw)}, Test={len(X_test_raw)}")
pca = PCA(n_components=n_qubits, random_state=123)
X_train_pca = pca.fit_transform(X_train_raw)
X_test_pca = pca.transform(X_test_raw)
print(f"PCA explained variance ratio sum: {pca.explained_variance_ratio_.sum():.4f}")
minmax = MinMaxScaler(feature_range=(0, np.pi))
X_train = minmax.fit_transform(np.array(X_train_pca, dtype=float))
X_test = np.clip(minmax.transform(np.array(X_test_pca, dtype=float)), 0, np.pi)
y_train = np.array(y_train); y_test = np.array(y_test)
n_features = X_train.shape[1]
print(f"Final shapes -> Train: {X_train.shape}, Test: {X_test.shape}")


## Variational Classifier


In [ ]:
dev = qml.device("default.qubit")


### Variational Data Prep


In [ ]:
X = np.array(X_train, dtype=float)
y = np.array(y_train, dtype=int)


In [ ]:
def layer(layer_weights):
    for wire in range(n_qubits):
        qml.Rot(*layer_weights[wire], wires=wire)
    for wires in ([0, 1], [1, 2], [2, 3], [3, 0]):
        qml.CNOT(wires=wires)


In [ ]:
def state_preparation(x):
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)


In [ ]:
@qml.qnode(dev)
def circuit(weights, x):
    state_preparation(x)
    for layer_weights in weights:
        layer(layer_weights)
    return [qml.expval(qml.PauliZ(0)), qml.expval(qml.PauliZ(1))]

def variational_classifier(weights, bias, x):
    return qml.math.stack(circuit(weights, x)) + bias

def cost(weights, bias, X, Y):
    predictions = qml.math.stack([variational_classifier(weights, bias, x) for x in X])
    targets = np.eye(n_classes)[np.array(Y, dtype=int)]
    return qml.math.mean((predictions - targets) ** 2)


In [ ]:
np.random.seed(123)
weights = 0.01 * np.random.randn(2, n_qubits, 3, requires_grad=True)
bias = np.zeros(n_classes, requires_grad=True)
opt = NesterovMomentumOptimizer(0.1)
batch_size = 5
start = time.time()
for it in range(100):
    batch_index = np.random.randint(0, len(X), (batch_size,))
    weights, bias = opt.step(cost, weights, bias, X=X[batch_index], Y=y[batch_index])
    if (it + 1) % 10 == 0 or it == 0:
        preds = [np.argmax(variational_classifier(weights, bias, x)) for x in X]
        current_cost = cost(weights, bias, X, y)
        acc = np.mean(np.array(preds) == np.array(y))
        print(f"Iter: {it+1:4d} | Cost: {current_cost:0.7f} | Accuracy: {acc:0.7f}")
train_time = time.time() - start
print(f"\nTraining time: {train_time:.4f} s")


In [ ]:
def predict(weights, bias, X_data):
    return np.array([np.argmax(variational_classifier(weights, bias, x)) for x in X_data])

def evaluate_split(weights, bias, X_data, y_data):
    preds = predict(weights, bias, X_data)
    y_true = np.array(y_data).astype(int)
    return (
        accuracy_score(y_true, preds),
        precision_score(y_true, preds, average="macro", zero_division=0),
        recall_score(y_true, preds, average="macro", zero_division=0),
        f1_score(y_true, preds, average="macro", zero_division=0),
        preds,
    )

start = time.time(); _ = predict(weights, bias, X_test); infer_time = time.time() - start
vqc_acc_tr, vqc_prec_tr, vqc_rec_tr, vqc_f1_tr, _ = evaluate_split(weights, bias, X, y)
vqc_acc, vqc_prec, vqc_rec, vqc_f1, y_test_pred = evaluate_split(weights, bias, X_test, y_test)
print("\n=== Train vs Test ===")
print(f"VQC Train - Acc: {vqc_acc_tr:.4f} | F1: {vqc_f1_tr:.4f}")
print(f"VQC Test  - Acc: {vqc_acc:.4f} | F1: {vqc_f1:.4f}")
print(f"Inference time: {infer_time:.4f} s")


In [ ]:
# Result Visualization (train vs test + paleta UNIR)
plt.rcParams.update({
    "font.sans-serif": ["Arial"],
    "axes.edgecolor": "#C0C0C8",
    "grid.color": "#C0C0C8",
    "grid.linestyle": "--",
})


train_color = "#0E4A75"
test_color = "#219BC3"
unir_gris = "#858591"

fig, axs = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
fig.suptitle("Variational Classifier — Ising phases", fontsize=18, y=1.02, color="#0E4A75")

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
train_vals = [vqc_acc_tr, vqc_prec_tr, vqc_rec_tr, vqc_f1_tr]
test_vals = [vqc_acc, vqc_prec, vqc_rec, vqc_f1]
x = np.arange(len(metrics))
width = 0.35

for i in range(len(metrics)):
    axs[0].bar(x[i] - width / 2, train_vals[i], width, color=train_color,
               label="Train" if i == 0 else "", zorder=3)
    axs[0].bar(x[i] + width / 2, test_vals[i], width, color=test_color,
               label="Test" if i == 0 else "", zorder=3)

axs[0].set_ylabel("Score")
axs[0].set_title("Train vs Test Metrics", color="#0E4A75")
axs[0].set_xticks(x)
axs[0].set_xticklabels(metrics)
axs[0].set_ylim(0, 1.35)
axs[0].legend(loc="upper right")
axs[0].grid(axis="y", zorder=0)

eff_labels = ["Training Time", "Inference Time"]
eff = [train_time, infer_time]
x_eff = np.arange(len(eff_labels))
axs[1].bar(x_eff, eff, 0.5, color="#219BC3", label="VQC", zorder=3)
axs[1].set_xticks(x_eff)
axs[1].set_xticklabels(eff_labels)
axs[1].set_ylabel("Time (s)")
axs[1].set_title("Computational Efficiency", color="#0E4A75")
max_time = max(eff)
axs[1].set_ylim(0, max_time * 1.35 if max_time > 0 else 1)
axs[1].legend(loc="upper right")
axs[1].grid(axis="y", zorder=0)
offset = max_time * 0.02 if max_time > 0 else 0.02
for i in range(2):
    axs[1].text(x_eff[i], eff[i] + offset, f"{eff[i]:.4f}s",
                ha="center", va="bottom", fontsize=9, color=unir_gris)

plt.tight_layout()
plt.show()


## Guardar resultados


In [ ]:
# Save metrics for IsingClassification comparison
# METRICS_PATH_OK: case_paths (no uses quantum/classical vars)
import json
from pathlib import Path
import sys

for _base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _lib = _base / "Notebooks" / "lib"
    if (_lib / "case_paths.py").is_file():
        _s = str(_lib)
        if _s not in sys.path:
            sys.path.insert(0, _s)
        break
else:
    raise FileNotFoundError("Notebooks/lib/case_paths.py not found")

import importlib
import case_paths as _case_paths
importlib.reload(_case_paths)
from case_paths import resolve_metrics_dir as _resolve_metrics_dir

def resolve_metrics_dir() -> Path:
    """Guarda en Notebooks/Classification/IsingClassification/metrics (helper compartido)."""
    return _resolve_metrics_dir("Classification", "IsingClassification")

metrics_dir = resolve_metrics_dir()



payload = {
    "model": "VQC",
    "family": "quantum",
    "protocol": {
        "dataset": "qspin-ising",
        "layouts": ["open/chain/1x4", "closed/chain/1x4"],
        "features": "ground_state Re||Im (32) -> PCA(4)",
        "label_rule": "y=(h > |J|) with J=-1 (1D TFIM critical field)",
        "n_features": int(n_features),
        "n_qubits": 4,
        "test_size": 0.4,
        "random_state": 123,
        "stratify": True,
        "scaler": "MinMaxScaler[0, pi]",
        "pca": "PCA(n_components=4)",
    },
    "train": {
        "accuracy": float(vqc_acc_tr),
        "precision": float(vqc_prec_tr),
        "recall": float(vqc_rec_tr),
        "f1": float(vqc_f1_tr),
        "auc": None,
    },
    "test": {
        "accuracy": float(vqc_acc),
        "precision": float(vqc_prec),
        "recall": float(vqc_rec),
        "f1": float(vqc_f1),
        "auc": None,
    },
    "train_time_s": float(train_time),
    "inference_time_s": float(infer_time),
}

out = metrics_dir / "vqc_ising.json"
out.write_text(json.dumps(payload, indent=2))
print("Saved:", out)


## Storytelling plots

`figures/`: PCA sin frontera + con frontera del **mismo** modelo ya entrenado.
Puntos test usan `*_preds` del notebook; la malla usa `modelo.predict` (sin reentrenar).


In [ ]:
# --- storytelling plots (auto) ---
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

for _base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _lib = _base / "Notebooks" / "lib"
    if (_lib / "storytelling.py").is_file():
        _s = str(_lib)
        if _s not in sys.path:
            sys.path.insert(0, _s)
        break
else:
    raise FileNotFoundError("Notebooks/lib/storytelling.py not found")

import importlib
import storytelling as _storytelling
importlib.reload(_storytelling)
from storytelling import resolve_figures_dir, plot_ising_story

figures_dir = resolve_figures_dir("Classification", "IsingClassification")
def _predict_fn(X):
    return predict(weights, bias, np.asarray(X, dtype=float))
predict_fn = _predict_fn
_raw = X_train_raw if "X_train_raw" in globals() else X_train
plot_ising_story(
    _raw, y_train,
    h_train=h_train if "h_train" in globals() else None,
    h_test=h_test if "h_test" in globals() else None,
    X_test_raw=X_test_raw if "X_test_raw" in globals() else None,
    figures_dir=figures_dir,
    stem="vqc",
    title_model="VQC",
    X_train=X_train,
    X_test=X_test,
    y_test=y_test,
    y_pred_test=y_test_pred,
    predict_fn=predict_fn,
)

plt.show()
